# Bonds & Futures Screening

This notebook demonstrates bond and futures screening using tvscreener.

**Screeners covered:**
- `BondScreener` - Government and corporate bonds (~200 fields)
- `FuturesScreener` - Futures contracts (~400 fields)

In [ ]:
from tvscreener import BondScreener, FuturesScreener
from tvscreener.field import BondField, FuturesField

---
# Bond Screener

## Basic Bond Screening

In [ ]:
bs = BondScreener()
df = bs.get()
print(f"Retrieved {len(df)} bonds with {len(df.columns)} columns")
df.head(10)

## Sort by Yield

In [ ]:
bs = BondScreener()

bs.sort_by(BondField.YIELD, ascending=False)
bs.set_range(0, 50)

bs.select(BondField.NAME, BondField.PRICE, BondField.YIELD, BondField.CHANGE_PERCENT)

df = bs.get()
print(f"Top {len(df)} bonds by yield")
df.head(15)

## Top Movers

In [ ]:
bs = BondScreener()

bs.sort_by(BondField.CHANGE_PERCENT, ascending=False)
bs.set_range(0, 30)

bs.select(BondField.NAME, BondField.PRICE, BondField.YIELD, BondField.CHANGE_PERCENT)

df = bs.get()
print(f"Top {len(df)} bond movers today")
df.head(15)

## Technical Analysis on Bonds

In [ ]:
bs = BondScreener()

bs.where(BondField.RELATIVE_STRENGTH_INDEX_14 < 40)

bs.select(BondField.NAME, BondField.PRICE, BondField.YIELD, BondField.RELATIVE_STRENGTH_INDEX_14)
bs.sort_by(BondField.RELATIVE_STRENGTH_INDEX_14, ascending=True)

df = bs.get()
print(f"Found {len(df)} bonds with RSI < 40")
df.head(15)

## Specific Bonds (US Treasuries)

In [ ]:
bs = BondScreener()

bs.symbols = {
    "query": {"types": []},
    "tickers": [
        "TVC:US02Y",  # 2-Year
        "TVC:US05Y",  # 5-Year
        "TVC:US10Y",  # 10-Year
        "TVC:US30Y",  # 30-Year
        "TVC:DE10Y",  # German 10-Year
        "TVC:GB10Y",  # UK 10-Year
        "TVC:JP10Y",  # Japan 10-Year
    ],
}

bs.select(BondField.NAME, BondField.PRICE, BondField.YIELD, BondField.CHANGE_PERCENT)

df = bs.get()
df

## Select All Bond Fields

In [ ]:
bs = BondScreener()
bs.select_all()
bs.set_range(0, 10)

df = bs.get()
print(f"Retrieved {len(df.columns)} columns for {len(df)} bonds")
df.head()

---
# Futures Screener

## Basic Futures Screening

In [ ]:
futs = FuturesScreener()
df = futs.get()
print(f"Retrieved {len(df)} futures with {len(df.columns)} columns")
df.head(10)

## Sort by Volume

In [ ]:
futs = FuturesScreener()

futs.sort_by(FuturesField.VOLUME, ascending=False)
futs.set_range(0, 50)

futs.select(FuturesField.NAME, FuturesField.PRICE, FuturesField.CHANGE_PERCENT, FuturesField.VOLUME)

df = futs.get()
print(f"Top {len(df)} futures by volume")
df.head(15)

## Top Gainers

In [ ]:
futs = FuturesScreener()

futs.where(FuturesField.CHANGE_PERCENT > 2)  # Up 2%+

futs.select(FuturesField.NAME, FuturesField.PRICE, FuturesField.CHANGE_PERCENT, FuturesField.VOLUME)
futs.sort_by(FuturesField.CHANGE_PERCENT, ascending=False)

df = futs.get()
print(f"Found {len(df)} futures up 2%+ today")
df.head(15)

## Technical Analysis on Futures

In [ ]:
futs = FuturesScreener()

futs.where(FuturesField.RELATIVE_STRENGTH_INDEX_14 < 30)  # Oversold

futs.select(
    FuturesField.NAME,
    FuturesField.PRICE,
    FuturesField.RELATIVE_STRENGTH_INDEX_14,
    FuturesField.CHANGE_PERCENT,
)
futs.sort_by(FuturesField.RELATIVE_STRENGTH_INDEX_14, ascending=True)

df = futs.get()
print(f"Found {len(df)} oversold futures (RSI < 30)")
df.head(15)

## Moving Average Analysis

In [ ]:
# Note: Field-to-field comparisons are NOT supported by TradingView API.
# Instead, retrieve data and filter with pandas:

futs = FuturesScreener()

futs.select(
    FuturesField.NAME,
    FuturesField.PRICE,
    FuturesField.SIMPLE_MOVING_AVERAGE_50,
    FuturesField.CHANGE_PERCENT,
)

df = futs.get()

# Filter for price above 50 SMA using pandas
above_sma = df[df["Price"] > df["SMA 50"]]
print(f"Found {len(above_sma)} futures above 50 SMA")
above_sma.head(15)

## High Volatility Futures

In [ ]:
futs = FuturesScreener()

futs.where(FuturesField.AVERAGE_TRUE_RANGE_14 > 0)

futs.select(
    FuturesField.NAME,
    FuturesField.PRICE,
    FuturesField.AVERAGE_TRUE_RANGE_14,
    FuturesField.VOLATILITY_D,
    FuturesField.CHANGE_PERCENT,
)
futs.sort_by(FuturesField.AVERAGE_TRUE_RANGE_14, ascending=False)
futs.set_range(0, 30)

df = futs.get()
print(f"Top {len(df)} futures by ATR")
df.head(15)

## Specific Futures Contracts

In [ ]:
futs = FuturesScreener()

futs.symbols = {
    "query": {"types": []},
    "tickers": [
        "CME:ES1!",  # E-mini S&P 500
        "CME:NQ1!",  # E-mini NASDAQ
        "CBOT:YM1!",  # E-mini Dow
        "COMEX:GC1!",  # Gold
        "COMEX:SI1!",  # Silver
        "NYMEX:CL1!",  # Crude Oil
        "NYMEX:NG1!",  # Natural Gas
    ],
}

futs.select(FuturesField.NAME, FuturesField.PRICE, FuturesField.CHANGE_PERCENT, FuturesField.VOLUME)

df = futs.get()
df

## Multi-Timeframe Analysis

In [ ]:
# Note: Field-to-field comparisons are NOT supported by TradingView API.
# For multi-timeframe analysis, retrieve all data and filter with pandas.

futs = FuturesScreener()

# 4-hour RSI moderate
rsi_4h = FuturesField.RELATIVE_STRENGTH_INDEX_14.with_interval("240")
futs.where(rsi_4h.between(40, 60))

futs.select(
    FuturesField.NAME,
    FuturesField.PRICE,
    FuturesField.SIMPLE_MOVING_AVERAGE_50,
    FuturesField.RELATIVE_STRENGTH_INDEX_14,
    rsi_4h,
    FuturesField.CHANGE_PERCENT,
)

df = futs.get()

# Filter for price above 50 SMA using pandas
multi_tf = df[df["Price"] > df["SMA 50"]]
print(f"Found {len(multi_tf)} multi-timeframe setups")
multi_tf.head(15)

## Select All Futures Fields

In [ ]:
futs = FuturesScreener()
futs.select_all()
futs.set_range(0, 10)

df = futs.get()
print(f"Retrieved {len(df.columns)} columns for {len(df)} futures")
df.head()